In [ ]:
try:
    import dlt
except ModuleNotFoundError:
    print(" This notebook is a DLT pipeline source.")
    print("   It cannot run interactively — the 'dlt' module is only available inside a pipeline.")
    print()
    print(" To run this code:")
    print("   1. Open Workflows → Lakeflow Declarative Pipelines")
    print("   2. Select pipeline: 'Salesforce Bronze Ingestion'")
    print("      Pipeline ID: 841459ab-105b-47c2-8688-abd14f0b76db")
    print("   3. Click 'Start' to trigger a pipeline update")
    print()
    print("   Target: data_engineering_workshop.sf_data")
    dbutils.notebook.exit("Run via pipeline, not interactively.")

In [ ]:
import requests
import pandas as pd
from pyspark.sql import functions as F
# =========================
# Configuration
# =========================
SCOPE = "salesforce-scope"
SF_DOMAIN = "login"   # "login" for prod, "test" for sandbox
SF_VERSION = "v60.0"

In [ ]:
# Salesforce helpers
# =========================
def _get_secret(key: str, default: str = "") -> str:
    try:
        return dbutils.secrets.get(SCOPE, key)
    except Exception:
        return default

def _sf_login():
    """Authenticate via Salesforce SOAP login (requires only username + password)."""
    username = _get_secret("sf-username")
    password = _get_secret("sf-password")
    token = _get_secret("sf-token", "")  # optional security token
    missing = []
    if not username:
        missing.append("sf-username")
    if not password:
        missing.append("sf-password")
    if missing:
        raise ValueError(f"Missing secrets in scope '{SCOPE}': {', '.join(missing)}")
    # SOAP login — no Connected App / client_id needed
    soap_url = f"https://{SF_DOMAIN}.salesforce.com/services/Soap/u/{SF_VERSION.lstrip('v')}"
    soap_body = f"""<?xml version="1.0" encoding="utf-8" ?>
    <env:Envelope xmlns:xsd="http://www.w3.org/2001/XMLSchema"
                  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
                  xmlns:env="http://schemas.xmlsoap.org/soap/envelope/">
      <env:Body>
        <n1:login xmlns:n1="urn:partner.soap.sforce.com">
          <n1:username>{username}</n1:username>
          <n1:password>{password}{token}</n1:password>
        </n1:login>
      </env:Body>
    </env:Envelope>"""
    headers = {"Content-Type": "text/xml; charset=utf-8", "SOAPAction": "login"}
    r = requests.post(soap_url, data=soap_body, headers=headers, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Salesforce SOAP login failed: {r.status_code} - {r.text}")
    # Parse session ID and server URL from SOAP response
    import xml.etree.ElementTree as ET
    root = ET.fromstring(r.text)
    ns = {
        "soapenv": "http://schemas.xmlsoap.org/soap/envelope/",
        "sf": "urn:partner.soap.sforce.com"
    }
    result = root.find(".//sf:loginResponse/sf:result", ns)
    if result is None:
        # Check for SOAP fault
        fault = root.find(".//soapenv:Body/soapenv:Fault/faultstring", ns)
        msg = fault.text if fault is not None else r.text
        raise RuntimeError(f"Salesforce login fault: {msg}")
    session_id = result.find("sf:sessionId", ns).text
    server_url = result.find("sf:serverUrl", ns).text
    # Extract instance URL from server URL
    from urllib.parse import urlparse
    parsed = urlparse(server_url)
    instance_url = f"{parsed.scheme}://{parsed.hostname}"
    return session_id, instance_url

def _sf_query_all(object_name: str):
    access_token, instance_url = _sf_login()
    headers = {"Authorization": f"Bearer {access_token}"}
    soql = f"SELECT FIELDS(ALL) FROM {object_name} LIMIT 200"
    url = f"{instance_url}/services/data/{SF_VERSION}/queryAll"
    params = {"q": soql}
    all_records = []
    while True:
        r = requests.get(url, headers=headers, params=params, timeout=120)
        if r.status_code != 200:
            raise RuntimeError(f"Query failed for {object_name}: {r.status_code} - {r.text}")
        data = r.json()
        records = data.get("records", [])
        for rec in records:
            rec.pop("attributes", None)
        all_records.extend(records)
        if data.get("done", True):
            break
        next_url = data.get("nextRecordsUrl")
        if not next_url:
            break
        url = f"{instance_url}{next_url}"
        params = None
    if not all_records:
        return spark.createDataFrame([], "Id string")
    pdf = pd.DataFrame(all_records)
    return spark.createDataFrame(pdf)

def _bronze_df(object_name: str):
    df = _sf_query_all(object_name)
    return (
        df.withColumn("_ingest_ts", F.current_timestamp())
          .withColumn("_source", F.lit("salesforce"))
          .withColumn("_object", F.lit(object_name))
    )

In [ ]:
@dlt.table(
    name="bronze_salesforce_account",
    comment="Raw Account data from Salesforce",
    table_properties={"quality": "bronze"}
)
def bronze_salesforce_account():
    return _bronze_df("Account")